In [1]:
from GroundTruth import loadSheet
import pandas as pd

industry_agnostic_indicators_sheet = loadSheet("1QoOHmD0nxb52BIVpKyniVdYej1W5o1-sNot7DpaBl2w", "IndustryAgnostricIndicators!A1:L")
industry_agnostic_units_to_convert = industry_agnostic_indicators_sheet[industry_agnostic_indicators_sheet['isUnitConversion'] == "TRUE"]
industry_agnostic_units_to_convert = industry_agnostic_units_to_convert[['IndicatorID', 'isOutcomeEffect']]

industry_specific_indicators_sheet = loadSheet("1QoOHmD0nxb52BIVpKyniVdYej1W5o1-sNot7DpaBl2w", "IndustrySpecificIndicators!A1:L")
industry_specific_indicators_sheet = industry_specific_indicators_sheet[industry_specific_indicators_sheet['isUnitConversion'] == "TRUE"]
industry_specific_indicators_sheet = industry_specific_indicators_sheet[['IndicatorID', 'isOutcomeEffect']]

all_indicators = pd.concat([industry_agnostic_units_to_convert, industry_specific_indicators_sheet])
outcome_effect_indicators = all_indicators[all_indicators['isOutcomeEffect'] == "TRUE"]['IndicatorID']
outcome_effect_indicators.head(10)

0                ghg_scope1
1         ghg_scope2_market
2                ghg_scope3
3                       nox
4                       sox
5     particulate_emissions
6    carbon_credits_offsets
7               ghg_removal
8       waste_non_hazardous
9           waste_hazardous
Name: IndicatorID, dtype: object

In [2]:
import mysql.connector
groundtruth_sheet_id = '18HCMbUmXcK9N2d4GziwUrHUgEHnc81ZH_4v-r-uJKcI' #Sheet with list of groundtruth sheets
big_dataset_range = "BigDataset!A1:C"

mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="MyN3wP4ssw0rd",
  database="democratizeesg"
)

mycursor = mydb.cursor()


sql_query = "SELECT company_name, year, indicator_id, not_disclosed, value FROM democratizeesg.big_dataset_consolidated;"
mycursor.execute(sql_query)
results = mycursor.fetchall()

columns = [col[0] for col in mycursor.description]
df_all_rows = pd.DataFrame(results, columns=columns)

all_companies = loadSheet(groundtruth_sheet_id, big_dataset_range)
years_to_collect = ["2020", "2021", "2022", "2023", "2024"]

panel = pd.DataFrame()

for index, row in all_companies.iterrows():
    company = row['Company']
    for year in years_to_collect:
        disclosed_indicators = df_all_rows.query(f"company_name == '{company}' and year == {year} and not_disclosed == 0")

        disclosed_outcome_effect_indicators = pd.merge(
            disclosed_indicators,
            outcome_effect_indicators,
            left_on='indicator_id',
            right_on='IndicatorID'
        )

        framework_indicator = disclosed_indicators.query("indicator_id == 'frameworks'")
        isGRI, isTCFD, isSASB, isCSRD = 0,0,0,0
        if len(framework_indicator) == 1:
            frameworks_string = framework_indicator.iloc[0]['value']
            if 'GRI' in frameworks_string or 'Global Reporting Initiative' in frameworks_string:
                isGRI = 1
            if 'TCFD' in frameworks_string or 'Task Force on Climate-Related Financial Disclosures' in frameworks_string:
                isTCFD = 1
            if 'SASB' in frameworks_string or 'Sustainability Accounting Standards Board' in frameworks_string:
                isSASB = 1
            if 'CSRD' in frameworks_string or 'ESRS' in frameworks_string:
                isCSRD = 1

        new_row = pd.DataFrame({'company_name': [company], 'year': [year], 'disclosed_number': [len(disclosed_indicators)],'disclosed_outcome_effect_number': [len(disclosed_outcome_effect_indicators)], 'isGRI': [isGRI], 'isTCFD': [isTCFD], 'isSASB': [isSASB], 'isCSRD': [isCSRD]})
        panel = pd.concat([panel, new_row], ignore_index=True)

In [11]:
import numpy as np

#panel = panel.set_index(['company_name', 'year'])
#years = panel.index.get_level_values('year').to_list()
#panel['year'] = pd.to_numeric(panel['year'])

panel_stats = panel.describe().T
panel_stats['mean'] = np.floor(panel_stats['mean'] * 100) / 100
panel_stats['std'] = np.floor(panel_stats['std'] * 100) / 100
panel_stats.drop(columns=['25%', '75%'], inplace=True)
panel_stats.rename(columns = {'50%':'median'}, inplace=True)
panel_stats
#panel.sample(10)
#panel.shape

,count,mean,std,min,median,max
disclosed_number,835.0,28.90,13.40,0.0,32.0,50.0
disclosed_outcome_effect_number,835.0,11.43,7.02,0.0,14.0,23.0
isGRI,835.0,0.68,0.46,0.0,1.0,1.0
isTCFD,835.0,0.73,0.44,0.0,1.0,1.0
isSASB,835.0,0.52,0.49,0.0,1.0,1.0
isCSRD,835.0,0.09,0.29,0.0,0.0,1.0


In [31]:
from linearmodels import PanelOLS



formula = 'disclosed_number ~ 1 + isGRI + isTCFD + isSASB + isCSRD + EntityEffects + TimeEffects'
mod = PanelOLS.from_formula(formula, data=panel)
res = mod.fit(cov_type="clustered", cluster_entity=True, )

print(results)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [32]:
print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:       disclosed_number   R-squared:                        0.4486
Estimator:                   PanelOLS   R-squared (Between):              0.5751
No. Observations:                 835   R-squared (Within):               0.4879
Date:                Sun, Nov 09 2025   R-squared (Overall):              0.5447
Time:                        07:55:35   Log-likelihood                   -2625.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      134.25
Entities:                         167   P-value                           0.0000
Avg Obs:                       5.0000   Distribution:                   F(4,660)
Min Obs:                       5.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             38.490
                            

In [35]:
formula = 'disclosed_outcome_effect_number ~ 1 + isGRI + isTCFD + isSASB + isCSRD + EntityEffects + TimeEffects'
mod = PanelOLS.from_formula(formula, data=panel)
res = mod.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)

print(res)

                                 PanelOLS Estimation Summary                                 
Dep. Variable:     disclosed_outcome_effect_number   R-squared:                        0.3600
Estimator:                                PanelOLS   R-squared (Between):              0.4703
No. Observations:                              835   R-squared (Within):               0.3943
Date:                             Sun, Nov 09 2025   R-squared (Overall):              0.4428
Time:                                     08:18:21   Log-likelihood                   -2175.8
Cov. Estimator:                          Clustered                                           
                                                     F-statistic:                      92.806
Entities:                                      167   P-value                           0.0000
Avg Obs:                                    5.0000   Distribution:                   F(4,660)
Min Obs:                                    5.0000          